In [1]:
# -*- coding: utf-8 -*-
import copy
from heapq import heappush, heappop

# Kích thước ma trận (4x4 cho 15-puzzle)
n = 4

# Hướng di chuyển của ô trống: Dưới, Trái, Trên, Phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class PriorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, key):
        heappush(self.heap, key)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return not self.heap

class Node:
    def __init__(self, parent, mats, empty_tile_posi, cost, levels, h_val):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.cost = cost      # f(n) = g(n) + h(n)
        self.levels = levels  # g(n)
        self.h_val = h_val    # h(n)

    # So sánh để Priority Queue ưu tiên f(n) nhỏ nhất
    def __lt__(self, nxt):
        return self.cost < nxt.cost

    @staticmethod
    def calculateManhattan(mats, final):
        """Tính tổng khoảng cách Manhattan của các ô so với vị trí đích"""
        dist = 0
        pos = {}
        for r in range(n):
            for c in range(n):
                pos[final[r][c]] = (r, c)

        for r in range(n):
            for c in range(n):
                val = mats[r][c]
                if val != 0:
                    target_r, target_c = pos[val]
                    dist += abs(r - target_r) + abs(c - target_c)
        return dist

    @staticmethod
    def printPath(root):
        if root is None:
            return
        Node.printPath(root.parent)
        print(f"--- Bước {root.levels} ---")
        print(f"g(n)={root.levels}, h(n)={root.h_val}, f(n)={root.cost}")
        for row in root.mats:
            print(" ".join(f"{val:2}" for val in row))
        print("-" * 20)

    @staticmethod
    def solve(initial, final):
        # Tự động tìm vị trí số 0 ban đầu
        start_empty_pos = None
        for r in range(n):
            for c in range(n):
                if initial[r][c] == 0:
                    start_empty_pos = [r, c]

        visited = set()
        pq = PriorityQueue()

        h_init = Node.calculateManhattan(initial, final)
        root = Node(None, initial, start_empty_pos, h_init, 0, h_init)
        pq.push(root)

        print("Đang tìm lời giải (vui lòng đợi nếu trạng thái phức tạp)...")

        while not pq.empty():
            minimum = pq.pop()

            # Nếu h(n) = 0 nghĩa là đã khớp hoàn toàn với đích
            if minimum.h_val == 0:
                Node.printPath(minimum)
                print(f"==> THÀNH CÔNG SAU {minimum.levels} BƯỚC!")
                return

            mats_tuple = tuple(map(tuple, minimum.mats))
            if mats_tuple in visited:
                continue
            visited.add(mats_tuple)

            # Khám phá 4 hướng di chuyển
            for i in range(4):
                new_pos = [
                    minimum.empty_tile_posi[0] + rows[i],
                    minimum.empty_tile_posi[1] + cols[i],
                ]

                if 0 <= new_pos[0] < n and 0 <= new_pos[1] < n:
                    # Tạo ma trận mới
                    new_mats = [row[:] for row in minimum.mats]
                    x1, y1 = minimum.empty_tile_posi
                    x2, y2 = new_pos
                    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

                    if tuple(map(tuple, new_mats)) not in visited:
                        h_val = Node.calculateManhattan(new_mats, final)
                        g_val = minimum.levels + 1
                        child = Node(minimum, new_mats, new_pos, g_val + h_val, g_val, h_val)
                        pq.push(child)

# --- CHẠY THỬ ---
if __name__ == "__main__":
    initial_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 0, 14, 15]
    ]

    target_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 0]
    ]

    Node.solve(initial_state, target_state)

Đang tìm lời giải (vui lòng đợi nếu trạng thái phức tạp)...
--- Bước 0 ---
g(n)=0, h(n)=2, f(n)=2
 1  2  3  4
 5  6  7  8
 9 10 11 12
13  0 14 15
--------------------
--- Bước 1 ---
g(n)=1, h(n)=1, f(n)=2
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14  0 15
--------------------
--- Bước 2 ---
g(n)=2, h(n)=0, f(n)=2
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
--------------------
==> THÀNH CÔNG SAU 2 BƯỚC!
